# Clase 10 · Explicabilidad e interpretabilidad: ¿por qué el modelo dice lo que dice?

**Machine Learning** · Programa de Especialización Data Scientist · Sesión 10 de 14

Docente: **Josef Renato Rodriguez**

---

## Dónde quedamos

En la Clase 9, XGBoost le ganó a la regresión logística en la cartera de tarjetas: **Gini de 0.677 en el OOT de 2026** contra
0.637, y el 75 % de los defaults de 2026 cayó en el grupo *Muy alto*. Sabemos **qué tan bien** ordena. Hoy preguntamos **por qué**.

Esa pregunta no es académica. La hacen:

- **El validador de modelos:** ¿el modelo usa las variables que tienen sentido de negocio, y en la dirección correcta?
- **El comité de riesgos:** ¿qué mueve el score? ¿qué pasa si sube la utilización de la cartera?
- **El cliente rechazado** (y el regulador detrás): ¿por qué a mí me pusieron en *Muy alto*?

## Qué vas a hacer hoy (primera hora)

| Bloque | Pregunta | Herramienta | Min |
|---|---|---|---|
| 1 | ¿Qué dice un modelo interpretable por diseño? | Coeficientes y *odds ratios* de la logística | 10 |
| 2 | ¿Qué variables pesan más en XGBoost? | Importancia nativa y **por permutación** | 15 |
| 3 | ¿Cómo cambia el riesgo cuando se mueve una variable? | **PDP** e **ICE** | 15 |
| 4 | ¿Por qué *este* cliente quedó en *Muy alto*? | **SHAP**, global y local | 20 |

La segunda hora es **Clustering · Parte 1**, en el notebook `Clase10_Clustering_Parte1.ipynb`.

Igual que siempre: completa las celdas **`TU CÓDIGO`** y ejecuta la **`VERIFICACIÓN`** de abajo.

---
## Celda 0 · Librerías, datos y el modelo campeón de la Clase 9

La celda descarga `cartera_tarjetas.csv` (carpeta `Clase_9/Datos` del repositorio) y **vuelve a entrenar el campeón de la Clase 9**
con la misma receta: XGBoost con árboles de 2 niveles y parada temprana en el test de 2025. Deja listas estas piezas:

- `ent`, `test`, `oot`: las tres muestras (2024 · 2025 · 2026) y sus targets `y_ent`, `y_test`, `y_oot`.
- `Xe`, `Xt`, `Xo`: las variables codificadas que ve XGBoost (`tipo_empleo` en tres columnas one-hot).
- `xgb`: el campeón. `predecir(df)` le pasa una base **sin codificar** y devuelve la probabilidad de default.
- `logit`: una regresión logística con las variables **estandarizadas**, para el Bloque 1.
- `corte_muy_alto`: la probabilidad desde la que un cliente cae en *Muy alto* (el percentil 80 de 2024).

In [ ]:
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
for paquete in ["xgboost", "shap"]:            # Colab trae los dos; si falta alguno, se instala
    try:
        __import__(paquete)
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)
from xgboost import XGBClassifier
import shap
import warnings
warnings.filterwarnings("ignore")

SEED = 42
NAVY, BLUE, MAG, GREEN, GRAY = "#0A2559", "#1A56E8", "#E6115E", "#12B886", "#5B6B87"
plt.rcParams.update({
    "figure.figsize": (7, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-3):
    if obtenido is None:
        print(f"[ ] {nombre}: todavía no calculaste nada")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}")
    print(f"     tu resultado: {float(obtenido):.4f}   |   esperado: {float(esperado):.4f}")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"     pista: {pista}")
    return bool(cond)

# ── Datos: la cartera de tarjetas de la Clase 9 ───────────────────────────────
ARCHIVO = "cartera_tarjetas.csv"
URL = "https://raw.githubusercontent.com/josefrodrim/Super_ML_course/refs/heads/main/Clase_9/Datos/cartera_tarjetas.csv"
if not os.path.exists(ARCHIVO):
    try:
        urllib.request.urlretrieve(URL, ARCHIVO)
        print(f"{ARCHIVO} descargado del repositorio del curso")
    except Exception as e:
        raise FileNotFoundError(f"No se pudo descargar ({e}). Sube {ARCHIVO} al panel de archivos de Colab y vuelve a ejecutar.")
cartera = pd.read_csv(ARCHIVO)

ID, TIEMPO, TARGET = "id_cliente", "periodo", "default_3m"
NUM = ["edad", "ingreso_mensual", "antiguedad_meses", "utilizacion_tc",
       "dias_atraso_max_6m", "num_consultas_3m", "ratio_deuda_ingreso"]
CAT = ["tipo_empleo"]
X = NUM + CAT

def codificar(df):
    """Para XGBoost: numéricas tal cual (los NaN se quedan) y tipo_empleo en one-hot."""
    d = df[NUM].copy()
    for v in ["dependiente", "independiente", "informal"]:
        d[f"empleo_{v}"] = (df["tipo_empleo"] == v).astype(int)
    return d

anio = cartera[TIEMPO].str[:4].astype(int)
ent, test, oot = cartera[anio == 2024], cartera[anio == 2025], cartera[anio == 2026]
y_ent, y_test, y_oot = ent[TARGET].values, test[TARGET].values, oot[TARGET].values
Xe, Xt, Xo = codificar(ent), codificar(test), codificar(oot)

# ── El campeón de la Clase 9: XGBoost con árboles de 2 niveles ────────────────
xgb = XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=2, subsample=0.8, colsample_bytree=0.8,
                    eval_metric="auc", early_stopping_rounds=100, random_state=SEED, n_jobs=1)
xgb.fit(Xe, y_ent, eval_set=[(Xt, y_test)], verbose=False)

def predecir(df):
    """Probabilidad de default de XGBoost para una base SIN codificar (con las columnas de X)."""
    return xgb.predict_proba(codificar(df))[:, 1]

corte_muy_alto = float(np.quantile(predecir(ent), 0.8))

# ── Una logística interpretable: variables estandarizadas, 'dependiente' como referencia ──
logit = make_pipeline(
    ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
                       ("cat", OneHotEncoder(drop=["dependiente"]), CAT)]),
    LogisticRegression(max_iter=2000)).fit(ent[X], y_ent)

auc_xgb, auc_log = roc_auc_score(y_test, predecir(test)), roc_auc_score(y_test, logit.predict_proba(test[X])[:, 1])
print(f"XGBoost: {xgb.best_iteration + 1} árboles de 2 niveles · AUC test {auc_xgb:.4f} · Gini OOT {2 * roc_auc_score(y_oot, predecir(oot)) - 1:.3f}")
print(f"Logística estandarizada · AUC test {auc_log:.4f}")
print(f"Un cliente entra a 'Muy alto' desde una probabilidad de {corte_muy_alto:.4f}")

---
# Bloque 1 · Interpretable por diseño: la logística  ·  10 min

La regresión logística se explica sola: cada variable tiene **un coeficiente** y ese coeficiente vale igual para todos los clientes.
Como estandarizamos las numéricas, el coeficiente dice cuánto cambia el *log-odds* de default cuando la variable sube
**una desviación estándar**. Y $e^{\beta}$, el **odds ratio**, lo dice en algo que entiende un comité:

$$\text{odds} = \frac{p}{1-p}, \qquad \text{OR} = e^{\beta} \;\Rightarrow\; \text{si OR} = 2, \text{ los odds se duplican.}$$

### Ejercicio 1.1 — La tabla de *odds ratios*

Saca los nombres con `logit[0].get_feature_names_out()` y los coeficientes con `logit[-1].coef_[0]`. Arma `tabla_or` con las
columnas `variable`, `coef` y `odds_ratio`, ordenada de mayor a menor `coef`.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
nombres = [n.replace("num__", "").replace("cat__", "") for n in logit[0].get_feature_names_out()]
coef = None          # los coeficientes del último paso del pipeline
tabla_or = None      # DataFrame con variable, coef y odds_ratio = np.exp(coef), ordenado por coef

In [ ]:
# ── VERIFICACIÓN 1.1 ──────────────────────────────────────────────────
def fila_or(var):
    if tabla_or is None:
        return None
    return tabla_or.set_index("variable").loc[var, "odds_ratio"]
r = [check("odds ratio de utilizacion_tc", fila_or("utilizacion_tc"), 2.019, tol=0.02),
     check("odds ratio de antiguedad_meses", fila_or("antiguedad_meses"), 0.827, tol=0.02),
     check_bool("la primera fila es utilizacion_tc", tabla_or is not None and tabla_or["variable"].iloc[0] == "utilizacion_tc")]
print()
print("Bloque 1 COMPLETO" if all(r) else "Revisa 1.1")

Cómo se lee:

- **Utilización:** una desviación estándar más (unos 23 puntos de uso de la línea) **duplica los odds** de default (OR = 2.02).
- **Atraso, deuda/ingreso y consultas** suben el riesgo (OR de 1.5, 1.4 y 1.4). **Antigüedad e ingreso** lo bajan (OR < 1).
- **Edad** no hace nada (OR = 1.00).

La fortaleza es también el límite: **el efecto es el mismo en todo el rango.** Para la logística, pasar de 10 % a 33 % de
utilización pesa igual que pasar de 70 % a 93 %. ¿Será cierto? XGBoost no tiene esa restricción, y eso es lo que le dio
los 4 puntos de Gini. El precio: ya no hay un coeficiente que leer. Los Bloques 2 a 4 son las herramientas para recuperarlo.

---
# Bloque 2 · ¿Qué variables pesan más? Importancia global  ·  15 min

### Demostración 2.1 — La importancia que trae XGBoost

XGBoost reporta tres importancias distintas, y no coinciden:

- **`weight`:** cuántas veces la variable se usa para partir un nodo.
- **`gain`:** cuánto mejora la pérdida, en promedio, cuando la usa.
- **`cover`:** cuántos clientes pasan por las partidas que la usan.

In [ ]:
# ── DEMOSTRACIÓN 2.1 ──────────────────────────────────────────────────
booster = xgb.get_booster()
nativa = pd.DataFrame({t: pd.Series(booster.get_score(importance_type=t)) for t in ["gain", "weight", "cover"]})
nativa = (nativa / nativa.sum()).sort_values("gain", ascending=False)
print(nativa.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, t, color in [(axes[0], "gain", BLUE), (axes[1], "weight", GRAY)]:
    s = nativa[t].sort_values()
    ax.barh(s.index, s.values, color=color)
    ax.set_title(f"importancia '{t}'"); ax.set_xlabel("participación")
plt.tight_layout(); plt.show()

Con `gain`, el atraso explica la mitad del modelo. Con `weight`, **la antigüedad queda tercera**: el modelo la usa mucho, pero
cada vez para partidas que mejoran poco. Una variable continua con muchos valores posibles ofrece muchos cortes y sale
inflada en `weight`. **¿Cuál creerle?** A ninguna del todo: las tres miden cómo está construido el árbol, no cuánto
depende la **predicción** de cada variable. Para eso está la permutación.

### Ejercicio 2.2 — Importancia por permutación

La idea: si **desordenas** una columna en el test (la barajas entre clientes), rompes su relación con el target. Si el AUC
cae mucho, el modelo dependía de esa variable. Si no cae, la variable sobraba.

Completa `importancia_permutacion(df, y, n_rep=5)`: para cada variable de `X`, copia `df`, baraja esa columna con
`rng.permutation`, predice con `predecir` y guarda **AUC original − AUC barajado**. Repite `n_rep` veces y devuelve un
DataFrame con la caída media y su desviación estándar, ordenado de mayor a menor caída.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
def importancia_permutacion(df, y, n_rep=5, semilla=SEED):
    rng = np.random.default_rng(semilla)
    auc_base = roc_auc_score(y, predecir(df))
    filas = []
    for var in X:
        caidas = []
        for _ in range(n_rep):
            d = df.copy()
            # baraja la columna var de d con rng.permutation(...)
            # calcula la caída: auc_base - AUC con la columna barajada
            pass
        filas.append({"variable": var, "caida_auc": np.mean(caidas) if caidas else None, "desv": np.std(caidas) if caidas else None})
    return pd.DataFrame(filas).set_index("variable")

perm = None   # importancia_permutacion(test, y_test), ordenada de mayor a menor caida_auc

In [ ]:
# ── VERIFICACIÓN 2.2 ──────────────────────────────────────────────────
def caida(var):
    if perm is None or perm["caida_auc"].isna().any():
        return None
    return perm.loc[var, "caida_auc"]
r = [check("caída de AUC al barajar dias_atraso_max_6m", caida("dias_atraso_max_6m"), 0.141, tol=0.01),
     check("caída de AUC al barajar utilizacion_tc", caida("utilizacion_tc"), 0.038, tol=0.006),
     check("caída de AUC al barajar edad", caida("edad"), 0.000, tol=0.002),
     check_bool("la primera es dias_atraso_max_6m", perm is not None and perm.index[0] == "dias_atraso_max_6m", "¿ordenaste con sort_values?")]
print()
print("2.2 OK" if all(r) else "Revisa 2.2")
if perm is not None and not perm["caida_auc"].isna().any():
    s = perm["caida_auc"].sort_values()
    plt.figure(figsize=(7, 4))
    plt.barh(s.index, s.values, xerr=perm.loc[s.index, "desv"], color=[MAG if v > 0.005 else GRAY for v in s.values])
    plt.xlabel("caída del AUC en el test 2025"); plt.title("Importancia por permutación")
    plt.tight_layout(); plt.show()

Si barajas el atraso, el AUC cae **14 puntos** (de 0.84 a 0.70). Si barajas la utilización, cae 4. Si barajas **la edad o el tipo
de empleo, no pasa nada**: el modelo casi no los usa, aunque `weight` decía que la edad aparecía en el 7 % de las partidas.

Dos cuidados:

1. **Se mide en el test, no en el entrenamiento.** En el entrenamiento, el modelo puede depender de variables que solo
   le sirvieron para memorizar.
2. **Variables correlacionadas se reparten la importancia.** Si dos variables dicen casi lo mismo, barajar una no duele
   porque la otra la cubre, y las dos parecen poco importantes. Antes de sacar una variable, mira con qué se correlaciona.

---
# Bloque 3 · ¿Cómo cambia el riesgo? PDP e ICE  ·  15 min

La importancia dice **cuánto** pesa una variable. No dice **en qué dirección** ni **con qué forma**. Para eso está el
**gráfico de dependencia parcial (PDP)**:

1. Toma la base completa del test.
2. Fija la variable en un valor, por ejemplo `utilizacion_tc = 0.5`, **para todos los clientes**, sin tocar lo demás.
3. Predice y promedia. Ese es el riesgo medio «si todos tuvieran 50 % de utilización».
4. Repite para cada valor de una grilla y une los puntos.

### Ejercicio 3.1 — El PDP de la utilización, a mano

Completa `dependencia_parcial(df, var, grilla)`: devuelve un array con la probabilidad media de `predecir` cuando `var` vale
cada valor de la grilla.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
def dependencia_parcial(df, var, grilla):
    medias = []
    for valor in grilla:
        d = df.copy()
        # fija la columna var en valor para TODOS los clientes, predice y guarda el promedio
        pass
    return np.array(medias)

grilla_util = np.round(np.arange(0, 1.16, 0.05), 2)
pdp_util = None   # dependencia_parcial(test, "utilizacion_tc", grilla_util)

In [ ]:
# ── VERIFICACIÓN 3.1 ──────────────────────────────────────────────────
ok_pdp = pdp_util is not None and len(pdp_util) == len(grilla_util)
r = [check("PDP con utilización = 0", pdp_util[0] if ok_pdp else None, 0.0256, tol=0.002),
     check("PDP con utilización = 1.0", pdp_util[20] if ok_pdp else None, 0.1805, tol=0.006),
     check_bool("el PDP nunca baja cuando sube la utilización", ok_pdp and bool(np.all(np.diff(pdp_util) >= -1e-6)))]
print()
print("3.1 OK" if all(r) else "Revisa 3.1")

### Demostración 3.2 — PDP e ICE de la utilización y del atraso

El PDP es un promedio, y un promedio esconde cosas. Las curvas **ICE** (*individual conditional expectation*) son el mismo
cálculo **sin promediar**: una curva por cliente. Si las curvas son paralelas, el efecto es igual para todos. Si se abren en
abanico, el efecto depende de otras variables del cliente.

In [ ]:
# ── DEMOSTRACIÓN 3.2 ──────────────────────────────────────────────────
def ice(df, var, grilla, n=150, semilla=SEED):
    muestra = df.sample(n, random_state=semilla)
    curvas = np.empty((n, len(grilla)))
    for j, valor in enumerate(grilla):
        d = muestra.copy(); d[var] = valor
        curvas[:, j] = predecir(d)
    return curvas

grilla_atr = np.arange(0, 90, 5)
casos = [("utilizacion_tc", grilla_util, "utilización de la línea"), ("dias_atraso_max_6m", grilla_atr, "días de atraso máximo en 6 meses")]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, (var, grilla, etiqueta) in zip(axes, casos):
    curvas = ice(test, var, grilla)
    for c in curvas:
        ax.plot(grilla, c, color=GRAY, alpha=0.15, lw=0.8)
    pdp = dependencia_parcial(test, var, grilla) if pdp_util is not None else curvas.mean(0)   # sin el 3.1, el promedio de la muestra
    ax.plot(grilla, pdp, color=MAG, lw=3, label="PDP (promedio de todos)")
    ax.set_ylim(0, 0.6); ax.set_xlabel(etiqueta); ax.set_ylabel("probabilidad de default")
    ax2 = ax.twinx(); cuentas, _, _ = ax2.hist(test[var], bins=30, color=BLUE, alpha=0.15)
    ax2.set_ylim(0, cuentas.max() * 4); ax2.set_yticks([]); ax2.grid(False)   # el histograma, abajo: dónde hay clientes
    ax.legend(loc="upper left")
axes[0].set_title("Utilización: el riesgo se dispara pasado 0.70"); axes[1].set_title("Atraso: el gran salto es de 0 a 1-5 días")
plt.tight_layout(); plt.show()

Lo que la logística no podía decir:

- **La utilización no es lineal.** Entre 0.15 y 0.45 el riesgo medio casi no se mueve (3.9 % a 4.7 %). Pasado **0.70 se
  dispara**: 8 % en 0.70, 11 % en 0.80, 18 % en 1.0. Eso es lo que vio XGBoost y no la logística.
- **El atraso pesa sobre todo en el primer escalón.** Pasar de 0 a 5 días multiplica el riesgo medio por casi 4 (2.5 % → 9.8 %).
  Después sigue subiendo, pero más despacio.
- **Las curvas son escalones**, no rectas: un modelo de árboles solo puede cambiar de valor en los cortes que aprendió.
- **Más allá de los datos, la curva es plana.** Hay muy pocos clientes con utilización sobre 1.0 (mira el histograma):
  el modelo no tiene cómo saber qué pasa ahí, y simplemente repite el último escalón.
- **Las ICE se abren en abanico**: el mismo salto de utilización le sube mucho más el riesgo a un cliente que ya tiene
  atraso. En probabilidad, los efectos se multiplican.

> **Cuidado con el PDP.** Fijar `utilizacion_tc = 1.0` a todos crea clientes que no existen (un cliente con 100 % de
> utilización y deuda/ingreso de 0.05). Si las variables están muy correlacionadas, el PDP promedia sobre combinaciones
> imposibles. La alternativa para ese caso son los gráficos **ALE**, que solo comparan vecinos.

---
# Bloque 4 · ¿Por qué *este* cliente? SHAP  ·  20 min

Las herramientas anteriores explican **el modelo en promedio**. Un cliente en *Muy alto* quiere saber por qué **él** está ahí.
**SHAP** (*SHapley Additive exPlanations*) reparte la predicción de cada cliente entre sus variables, con una idea de la
teoría de juegos: cuánto aporta cada «jugador» en promedio, sobre todos los órdenes posibles en que se suman al equipo.

La propiedad que importa es la **aditividad**:

$$\underbrace{f(x)}_{\text{log-odds del cliente}} \;=\; \underbrace{\phi_0}_{\text{valor base}} \;+\; \sum_{j} \phi_j$$

- $\phi_0$ es la predicción media del modelo (en *log-odds*).
- $\phi_j$ es lo que **la variable $j$ de este cliente** empuja la predicción hacia arriba o hacia abajo.

Para modelos de árboles, `shap.TreeExplainer` calcula los $\phi_j$ **exactos** y rápido. En XGBoost, la salida está en
*log-odds*, no en probabilidad.

In [ ]:
# ── DEMOSTRACIÓN 4.0 ──────────────────────────────────────────────────
explainer = shap.TreeExplainer(xgb)
sv = explainer.shap_values(Xt)                          # una fila por cliente del test, una columna por variable
base = float(np.ravel(explainer.expected_value)[0])     # el valor base phi_0, en log-odds
print(f"SHAP del test: {sv.shape[0]:,} clientes × {sv.shape[1]} columnas".replace(",", " "))
print(f"Valor base: {base:.4f} en log-odds")

### Ejercicio 4.1 — Comprueba la aditividad

1. `margen_rec`: el valor base más la suma de los SHAP de cada cliente (`sv.sum(axis=1)`).
2. `margen_xgb`: el log-odds que da el modelo, con `xgb.predict(Xt, output_margin=True)`.
3. `diferencia_max`: la mayor diferencia absoluta entre los dos.
4. `p_base`: el valor base pasado a probabilidad, $1 / (1 + e^{-\phi_0})$.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
margen_rec = None       # base + suma de los SHAP de cada cliente
margen_xgb = None       # xgb.predict(Xt, output_margin=True)
diferencia_max = None   # la mayor diferencia absoluta entre los dos
p_base = None           # el valor base en probabilidad

In [ ]:
# ── VERIFICACIÓN 4.1 ──────────────────────────────────────────────────
r = [check_bool("base + Σ SHAP reproduce el log-odds del modelo", diferencia_max is not None and diferencia_max < 1e-3,
                "suma sv por filas (axis=1) y compara con output_margin=True"),
     check("valor base en probabilidad", p_base, 0.0598, tol=0.002)]
print()
print("4.1 OK" if all(r) else "Revisa 4.1")

El valor base, **6.0 %**, es el punto de partida de todos los clientes antes de mirar sus variables. Es casi la tasa de
default del entrenamiento (6.1 %); no coinciden exacto porque SHAP promedia en log-odds y recién después se convierte. Y la suma cuadra hasta el sexto decimal. Esa es la gran ventaja de SHAP sobre otras explicaciones locales:
**no es una aproximación de la predicción, es la predicción repartida.**

### Ejercicio 4.2 — Importancia global con SHAP

La importancia global de SHAP es el **promedio del valor absoluto** de los SHAP de cada variable: cuánto mueve, en promedio,
la predicción de un cliente. Arma `imp_shap`, una Series con la participación de cada variable (que sume 1), **juntando las
tres columnas de empleo** en una sola, `tipo_empleo`.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
media_abs = None   # pd.Series con el promedio de |SHAP| de cada columna de Xt
# junta las tres columnas empleo_* en una sola 'tipo_empleo' y bórralas
imp_shap = None    # participación de cada variable (media_abs / su suma), de mayor a menor

In [ ]:
# ── VERIFICACIÓN 4.2 ──────────────────────────────────────────────────
def part(var):
    return None if imp_shap is None else imp_shap.get(var)
r = [check("participación de dias_atraso_max_6m", part("dias_atraso_max_6m"), 0.459, tol=0.01),
     check("participación de utilizacion_tc", part("utilizacion_tc"), 0.200, tol=0.01),
     check("participación de tipo_empleo (las tres columnas)", part("tipo_empleo"), 0.013, tol=0.005),
     check_bool("suma 1", imp_shap is not None and abs(imp_shap.sum() - 1) < 1e-6)]
print()
print("4.2 OK" if all(r) else "Revisa 4.2")

### Demostración 4.3 — El gráfico de enjambre (*beeswarm*)

Cada punto es un cliente. La posición horizontal es su SHAP y el color es el valor de la variable (rojo alto, azul bajo).
En un solo gráfico se ve la importancia **y la dirección** del efecto.

In [ ]:
# ── DEMOSTRACIÓN 4.3 ──────────────────────────────────────────────────
shap.summary_plot(sv, Xt, show=False, plot_size=(9, 5))
plt.title("SHAP en el test 2025 · cada punto es un cliente")
plt.tight_layout(); plt.show()

- **Atraso:** los rojos (mucho atraso) están todos a la derecha, y la nube azul de los que nunca se atrasan, a la izquierda.
- **Utilización:** igual, y con una cola larga a la derecha: la utilización muy alta puede sumar hasta 1.6 de log-odds.
- **Antigüedad e ingreso:** los rojos a la izquierda. Más antigüedad e ingreso, menos riesgo. La misma dirección de la logística.
- **Edad y empleo:** pegados al cero. El mismo veredicto de la permutación, por otro camino.

### Ejercicio 4.4 — Los motivos de un cliente en *Muy alto*

El cliente **103883**, en su foto de **2025-01**. Busca su fila en el test, su probabilidad y sus SHAP, y arma `motivos`:
las **tres variables que más le suben el riesgo** (SHAP positivo más alto), con su valor y su SHAP.

Pista: `i = np.flatnonzero((test[ID] == 103883) & (test[TIEMPO] == "2025-01"))[0]` te da la posición en `Xt` y en `sv`.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
i = None            # la posición del cliente en Xt (usa la pista)
p_cliente = None    # su probabilidad de default
shap_cliente = None # pd.Series con los SHAP de su fila, con los nombres de las columnas de Xt
motivos = None      # DataFrame con 'valor' y 'shap' de las 3 variables con SHAP positivo más alto

In [ ]:
# ── VERIFICACIÓN 4.4 ──────────────────────────────────────────────────
r = [check("probabilidad del cliente 103883", p_cliente, 0.364, tol=0.01),
     check_bool("está en Muy alto", p_cliente is not None and p_cliente >= corte_muy_alto),
     check_bool("el primer motivo es utilizacion_tc", motivos is not None and motivos.index[0] == "utilizacion_tc"),
     check_bool("el segundo motivo es dias_atraso_max_6m", motivos is not None and len(motivos) > 1 and motivos.index[1] == "dias_atraso_max_6m")]
print()
print("4.4 OK" if all(r) else "Revisa 4.4")

In [ ]:
# ── DEMOSTRACIÓN 4.5 · EL WATERFALL DEL CLIENTE ───────────────────────
if i is not None:
    caso = shap.Explanation(values=sv[i], base_values=base, data=Xt.iloc[i].values, feature_names=list(Xt.columns))
    shap.plots.waterfall(caso, max_display=10, show=False)
    plt.title("Cliente 103883 · 2025-01 · de 6.0 % a 36.4 %")
    plt.tight_layout(); plt.show()

Cómo se le explica a un comité, o al cliente:

> El cliente parte del riesgo medio de la cartera (6.0 %). **Usa el 89 % de su línea**, lo que suma 1.16 de log-odds;
> **tuvo un atraso de 13 días** en los últimos seis meses (+0.95), y **lo consultaron dos veces** en la central de riesgos
> (+0.28). Su antigüedad de 69 meses le baja un poco el riesgo (−0.14). Termina en **36.4 %**, seis veces la media.

Fíjate en el **ingreso vacío**: suma +0.09. XGBoost aprendió que **no declarar ingreso** es en sí una señal de riesgo
(los vacíos son más comunes en informales). Es una explicación legítima, pero un validador la va a preguntar.

Los tres primeros motivos son los **códigos de motivo** (*reason codes*) que un banco guarda junto con cada decisión.
**Ojo:** los SHAP están en log-odds. Se pueden sumar, pero no se leen como «puntos de probabilidad». Para hablar en
probabilidad, se convierte el total al final, como hicimos aquí.

### Demostración 4.6 — Dependencia SHAP: el PDP de cada cliente

El gráfico de dependencia pone el valor de la variable en el eje x y su SHAP en el eje y: es la versión SHAP del PDP,
con un punto por cliente.

In [ ]:
# ── DEMOSTRACIÓN 4.6 ──────────────────────────────────────────────────
j = list(Xt.columns).index("utilizacion_tc")
plt.figure(figsize=(8, 4.4))
sc = plt.scatter(Xt["utilizacion_tc"], sv[:, j], c=np.minimum(Xt["dias_atraso_max_6m"], 30), cmap="coolwarm", s=6, alpha=0.6)
plt.colorbar(sc, label="días de atraso (tope 30)")
plt.axhline(0, color=GRAY, lw=1)
plt.xlabel("utilizacion_tc"); plt.ylabel("SHAP de utilizacion_tc (log-odds)")
plt.title("Bajo 0.45 la utilización resta riesgo; sobre 0.70 lo suma, y mucho")
plt.tight_layout(); plt.show()

La curva cruza el cero cerca de **0.45-0.50**: debajo, la utilización **baja** el riesgo del cliente frente a la media; arriba,
lo sube. La forma es la del PDP del Bloque 3, pero ahora cada punto es un cliente real, sin inventar combinaciones.

Como los árboles tienen 2 niveles, el modelo solo puede aprender interacciones de **dos variables a la vez**. Aquí casi no
hay: los colores se mezclan a la misma altura. El efecto de la utilización es casi el mismo con o sin atraso **en log-odds**.
El abanico de las ICE del Bloque 3 venía de convertir a probabilidad, donde los efectos se multiplican.

### Demostración 4.7 — SHAP por año: ¿el modelo sigue mirando lo mismo?

Una aplicación de monitoreo: la participación de cada variable en 2024, 2025 y 2026. Si en el OOT una variable gana o
pierde peso de golpe, cambió **la población** (no el modelo, que es el mismo).

In [ ]:
# ── DEMOSTRACIÓN 4.7 ──────────────────────────────────────────────────
def participacion(Xm):
    s = pd.Series(np.abs(explainer.shap_values(Xm)).mean(axis=0), index=Xm.columns)
    emp = [c for c in Xm.columns if c.startswith("empleo_")]
    s["tipo_empleo"] = s[emp].sum()
    s = s.drop(emp)
    return s / s.sum()

por_anio = pd.DataFrame({"2024": participacion(Xe), "2025": participacion(Xt), "2026": participacion(Xo)})
print(por_anio.sort_values("2024", ascending=False).round(3).to_string())

Las participaciones casi no se mueven (el atraso pasa de 45.9 % a 45.2 %, la antigüedad de 6.4 % a 7.6 %). Encaja con lo
de la Clase 9: en 2026 **el modelo ordena igual y mira lo mismo**; lo que falló fue el nivel de las probabilidades.

---
# Cierre de la primera hora

### Los números del día

| Herramienta | Qué respondió |
|---|---|
| *Odds ratio* (logística) | +1 desv. est. de utilización duplica los odds (OR 2.02) |
| Permutación | Sin atraso el AUC cae 14 puntos; sin edad o empleo, nada |
| PDP | Utilización: 3.9 % en 0.20, 18 % en 1.0, y el salto empieza pasado 0.70 |
| SHAP global | Atraso 46 %, utilización 20 %, deuda/ingreso 10 %, consultas 9 % |
| SHAP local | Cliente 103883: de 6.0 % a 36.4 % por utilización, atraso y consultas |

### Checklist de salida

- [ ] Distingo **interpretable por diseño** (la logística) de **explicar un modelo complejo** (XGBoost + herramientas).
- [ ] Distingo explicación **global** (cómo funciona el modelo) de **local** (por qué este cliente).
- [ ] Sé por qué la importancia `weight` de XGBoost puede engañar, y calculo la importancia **por permutación** en el test.
- [ ] Construyo un **PDP** a mano, leo las **ICE** y conozco su límite con variables correlacionadas.
- [ ] Compruebo la **aditividad** de SHAP y sé que en XGBoost está en log-odds.
- [ ] Saco los **tres motivos** de un cliente en *Muy alto* y los explico en lenguaje de negocio.

### Preguntas tipo examen

**1.** La importancia `weight` de XGBoost pone a la antigüedad en tercer lugar y la permutación casi en el último. ¿A cuál creerle para decidir si sacarla del modelo?

> **Respuesta.** A la **permutación en el test**. `weight` cuenta cuántas veces se usa la variable para partir, y las variables
> continuas con muchos cortes posibles salen infladas. La permutación mide cuánto empeora la predicción sin ella.

**2.** Al barajar `ingreso_mensual` el AUC casi no cae. Un compañero propone sacarla. ¿Qué revisarías antes?

> **Respuesta.** Si está **correlacionada** con otra variable que la cubre (aquí, `ratio_deuda_ingreso` se calcula con el
> ingreso). Barajar una de dos variables redundantes no duele, aunque la información sí importe.

**3.** El SHAP de la utilización de un cliente es +1.16. ¿Significa que la utilización le sube 116 puntos de probabilidad?

> **Respuesta.** No. En XGBoost los SHAP están en **log-odds**. Se suman al valor base (−2.76) y el total se convierte a
> probabilidad al final. Un mismo +1.16 sube más la probabilidad de un cliente que ya está alto que la de uno que está bajo.

**4.** ¿Por qué el PDP de la utilización es plano entre 1.0 y 1.15?

> **Respuesta.** Porque casi no hay clientes ahí: el árbol no aprendió cortes en esa zona y repite el último escalón.
> No significa que el riesgo deje de subir; significa que **el modelo no sabe**. No se extrapola.

**5.** El validador pregunta si el modelo «discrimina por edad». ¿Qué le muestras?

> **Respuesta.** Tres evidencias: la permutación (barajar la edad no cambia el AUC), la participación SHAP (0.4 %) y su
> gráfico SHAP pegado al cero. El modelo casi no usa la edad. Igual, si la variable no debe usarse por política, se saca.

### Lo que sigue

**Segunda hora · Clustering, parte 1** · notebook `Clase10_Clustering_Parte1.ipynb`. Cambiamos de pregunta: ya no hay target.
Con los mismos clientes de la cartera, buscamos **segmentos de comportamiento** con K-Means.